<a href="https://colab.research.google.com/github/rishiks29/teen_vaccine_UTD_model/blob/main/NIS_TEEN_Final_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NIS-TEEN Modeling Study
# Author: Rishik

# Step 1
# IMPORTS NEEDED FOR THE MODELING
import numpy as np
import pandas as pd
import pandas.api.types as ptypes
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from google.colab import files

# IMPORT RAW FILES
# USE_WEIGHTS = True
# USE_CLASS_WEIGHT = True
USE_CALIBRATION = True
RARE_CAT_MIN_CT = 50
TEST_SIZE = 0.30
RANDOM_STATE = 1234

# # Grid for elastic-net logistic
# C_GRID = list(np.logspace(-2, 1, 8)) # 0.01..10
# L1R_GRID = [0.2, 0.5, 0.8] # l1_ratio

# UPLOAD & LOAD
print("Please upload nisteenpuf23.csv")
_ = files.upload()
file_path = "/content/nisteenpuf23.csv"
df = pd.read_csv(file_path, low_memory=False)
print("Loaded:", df.shape)

In [ ]:
# Check frequencies and missingness
print( 'P_UTD13212 frequencies:' )
print(df[ 'P_UTD13212' ].value_counts(dropna=False))
print( '\nP_UTDHPV frequencies:' )
print(df[ 'P_UTDHPV' ].value_counts(dropna=False))
utd_utd_count = df[ (df[ 'P_UTD13212' ] == 'UTD' ) & ( df[ 'P_UTDHPV' ] == 'UTD' )].shape[0]
print( f"Number of rows where P_UTD13212 is 'UTD' and P_UTDHPV is 'UTD': {utd_utd_count}")

In [ ]:
# Adequate provider Data Filter
pdat2_raw = df[ "PDAT2"].astype(str).str.strip().str.upper()
print( "PDAT2 values (top):\n", pdat2_raw.value_counts().head(10) )

adequate_mask = pdat2_raw.str.contains("HAS ADEQUATE PROVIDER DATA", na=False)
df["PDAT2_FLAG"] = adequate_mask.astype(int)
df = df[df["PDAT2_FLAG"] == 1].copy()
print("After PDAT2 filter:", df.shape)

# Define Study Outcome
def utd_flag(series):
    s = series.astype(str).str.strip().str.upper()
    if s.isin([ "UTD","NOT UTD" ]).any():
        return (s == "UTD").astype(int)
    return (pd.to_numeric(series, errors="coerce") > 0).fillna(0).astype(int)

for col in [ "P_UTD13212","P_UTDFLU2223","P_UTDHPV","P_UTDCOV_FULL" ]:
    if col not in df.columns:
        df[col] = np.nan

df["UTD13212_flag"] = utd_flag(df["P_UTD13212"])
df["UTDHPV_flag"]   = utd_flag(df["P_UTDHPV"])
df["UTDFLU_flag"]   = utd_flag(df["P_UTDFLU2223"])
df["UTDCOV_flag"]   = utd_flag(df["P_UTDCOV_FULL"])

# routine series + HPV, excluding flu/COVID
df[ "noFLUCOV_vax_UTD" ] = ((df[ "UTD13212_flag" ]==1) & (df[ "UTDHPV_flag" ]==1)).astype(int)
y_col = "noFLUCOV_vax_UTD"

print("Outcome definition: y=1 means UTD routine+HPV (we are excluding flu/COVID).")
print("noFLUCOV_vax_UTD counts:\n", df[y_col].value_counts())
print("UTD prevalence:", df[y_col].mean().round(3))

# Check Weights
WEIGHT_COL = "PROVWT_C"
if USE_WEIGHTS:
    assert WEIGHT_COL in df.columns, f"{WEIGHT_COL} missing but USE_WEIGHTS=True"
    df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors="coerce")
    print("Weight summary:\n", df[WEIGHT_COL].describe())

# Traning vs Validation Splits
train_df, valid_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True, stratify=df[y_col]
)

print(f"Train: {len(train_df)}  Valid: {len(valid_df)}")


In [ ]:
unweighted_rate = df["noFLUCOV_vax_UTD"].mean()
weighted_rate = (df["noFLUCOV_vax_UTD"] * df["PROVWT_C"]).sum() / df["PROVWT_C"].sum()

print(f"Unweighted: {unweighted_rate:.3f}")
print(f"Weighted: {weighted_rate:.3f}")

In [ ]:
# Normalize generic purely Yes/No fields
YN_MAP = {"YES":1,"Y":1,"TRUE":1,"T":1,"1":1,"NO":0,"N":0,"FALSE":0,"F":0,"0":0}
def normalize_yes_no(df_):
    df_ = df_.copy()
    for c in df_.select_dtypes(include="object").columns:
        s = df_[c].astype(str).str.strip().str.upper()
        if set(s.unique()) <= set(YN_MAP.keys()):
            df_[c] = s.map(YN_MAP).astype(float)
    return df_
train_df = normalize_yes_no(train_df)
valid_df = normalize_yes_no(valid_df)

# List of Predictors
base_vars = [
    "AGE","AGEGRP_M_I","ASTHMA","C1R","C5R", "CEN_REG","EDUC1","EDUC_TR",
    "INCPORAR_I","INCPOV1","INCQ298A","I_HISP_K","RACE_K","SEX","LANGUAGE",
    "MARITAL2","MOBIL_I","RACEETHK","RENT_OWN","STATE","WELLCHILD","VISITS",
    "CHILDNM","CKUP_11_12","CPOX_HAD","CPOX_AGE","HPVI_RECOM",
    "IMM_ANY","FACILITY","NOSCHOOLR"
]
risk_like = [c for c in train_df.columns if c.startswith("RISK_")]
X_cols = [c for c in dict.fromkeys(base_vars + risk_like) if c in train_df.columns]

# Leakage Guarding
leak_patterns = [
    "_UTD","_DAGE","_MAGE","_SAGE","_AGE1","_AGE2",
    "_DOSE","_SHOT","_RECV","_VACC",
    "HPV_","MCV_","MEN_","HEPA_","HEPB_","VRC_",
    "_IMMR","_DT_"
]
leak_cols   = [c for c in X_cols if any(p in c.upper() for p in leak_patterns)]
weight_like = {"PROVWT_C","PROVWT_C_TERR","RDDWT_C"}
id_like     = {"SEQNUMT"}
admin_like  = {"N_PRVR", "VFC_ORDER_MISSING"}
extra_exclude = set(leak_cols) | weight_like | id_like | admin_like | {
    "PDAT2","PDAT2_FLAG",
    "P_UTD13212","P_UTDFLU2223","P_UTDHPV","P_UTDCOV_FULL",
    "UTD13212_flag","UTDHPV_flag","UTDFLU_flag","UTDCOV_flag", y_col
}
X_cols = [c for c in X_cols if c not in extra_exclude]
print(f"[Leakage guard] Dropped {len(extra_exclude)} cols. Using {len(X_cols)} predictors.")
if not X_cols:
    raise ValueError("No predictors remain after leakage guard.")

# Forcing Categorical Variables
force_cat = [
    "STATE","CEN_REG","AGEGRP_M_I","EDUC1","EDUC_TR","INCPOV1","INCPORAR_I",
    "INCQ298A","RACE_K","RACEETHK","SEX","LANGUAGE","MARITAL2","MOBIL_I",
    "RENT_OWN","WELLCHILD","VISITS","CKUP_11_12","CPOX_HAD","CPOX_AGE",
    "HPVI_RECOM","IMM_ANY","FACILITY","NOSCHOOLR","C1R","C5R","CHILDNM"
]
for c in force_cat:
    if c in train_df.columns:
        train_df[c] = train_df[c].astype("object")
        valid_df[c] = valid_df[c].astype("object")

# Collpasing Categorical Variables where needed
categorical_features = [c for c in X_cols if not ptypes.is_numeric_dtype(train_df[c])]
numeric_features     = [c for c in X_cols if ptypes.is_numeric_dtype(train_df[c])]

def collapse_rare_cats_train_apply(train_, valid_, cols, min_count=50):
    for c in cols:
        s_tr = train_[c].astype(str).fillna("__NA__")
        vc = s_tr.value_counts(dropna=False)
        rare = set(vc[vc < min_count].index)
        for df_ in (train_, valid_):
            s = df_[c].astype(str).fillna("__NA__")
            df_.loc[:, c] = s.where(~s.isin(rare), "__OTHER__")

if RARE_CAT_MIN_CT and categorical_features:
    collapse_rare_cats_train_apply(train_df, valid_df, categorical_features, RARE_CAT_MIN_CT)

# Align levels in Valid and Train
def align_categories_to_train(train_, valid_, cols):
    for c in cols:
        train_vals = set(train_[c].astype(str).fillna("__NA__").unique())
        valid_vals = valid_[c].astype(str).fillna("__NA__")
        valid_[c] = valid_vals.where(valid_vals.isin(train_vals), "__OTHER__")

if categorical_features:
    align_categories_to_train(train_df, valid_df, categorical_features)

# numeric coercion & fill
for c in numeric_features:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    valid_df[c] = pd.to_numeric(valid_df[c], errors="coerce")
train_df[numeric_features] = train_df[numeric_features].fillna(0)
valid_df[numeric_features] = valid_df[numeric_features].fillna(0)

# Pre-Processing features
transformers = []
if categorical_features:
    transformers.append(
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    )
transformers.append(
    ("num", Pipeline(steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value=0)),
        ("scale", StandardScaler())
    ]), numeric_features)
)
preprocess = ColumnTransformer(transformers)
print("Predictors (X_cols):")
print(X_cols)

In [ ]:
# Add QC Checks
bad = []
for c in numeric_features:
    if train_df[c].astype(str).str.upper().isin(["YES","NO","DON'T KNOW","MISSING"]).any():
        bad.append(c)
print("Numeric columns with string-like values:", bad)

Xtr = preprocess.fit_transform(train_df[X_cols])
Xva = preprocess.transform(valid_df[X_cols])
print(Xtr.shape, Xva.shape)

In [ ]:
# Install Lazypredict package
!pip install lazypredict

In [ ]:
# MODEL SCREENING — LazyClassifier without hyperparameter Tuning
import numpy as np
import pandas as pd
from lazypredict.Supervised import LazyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score

# Prepare X/y
X_train = train_df[X_cols].copy()
y_train = train_df[y_col].astype(int).values

X_valid = valid_df[X_cols].copy()
y_valid = valid_df[y_col].astype(int).values

# Fit pre-processing on TRAIN only
X_train_t = preprocess.fit_transform(X_train)
X_valid_t = preprocess.transform(X_valid)

# Run LazyClassifier (default models+LR Models)
lazy = LazyClassifier(verbose=0, ignore_warnings=True)
models, predictions = lazy.fit(X_train_t, X_valid_t, y_train, y_valid)

def eval_model_row(name, clf):
    clf.fit(X_train_t, y_train)
    yhat = clf.predict(X_valid_t)

    auc = np.nan
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X_valid_t)[:, 1]
        auc = roc_auc_score(y_valid, p)

    return pd.Series({
        "Accuracy": accuracy_score(y_valid, yhat),
        "Balanced Accuracy": balanced_accuracy_score(y_valid, yhat),
        "ROC AUC": auc,
        "F1 Score": f1_score(y_valid, yhat),
        "Time Taken": np.nan
    }, name=name)

# Simple Logistic (L2)
lr_l2 = LogisticRegression(
    solver="lbfgs",
    penalty="l2",
    C=1.0,
    max_iter=5000
)

# LASSO Logistic (L1)
lr_l1 = LogisticRegression(
    solver="saga",
    penalty="l1",
    C=1.0,
    max_iter=5000,
    random_state=42
)

extra_rows = pd.DataFrame([
    eval_model_row("LogisticRegression_L2(Simple)", lr_l2),
    eval_model_row("LogisticRegression_L1(LASSO)",  lr_l1),
])

# Combine + rank
leaderboard = pd.concat([models, extra_rows], axis=0, sort=False)
leaderboard = leaderboard.sort_values(
    ["Accuracy", "F1 Score", "ROC AUC"],
    ascending=False
)

print("\n=== MODEL SCREENING (VALID, default hyperparameters) ===")
display(leaderboard.head(25))

print("\nTop by Accuracy:")
display(leaderboard.sort_values("Accuracy", ascending=False).head(5))

print("\nTop by F1:")
display(leaderboard.sort_values("F1 Score", ascending=False).head(5))

print("\nTop by AUROC:")
display(leaderboard.sort_values("ROC AUC", ascending=False).head(5))

print("\nLogistic variants only:")
display(
    leaderboard.loc[
        leaderboard.index.str.contains("Logistic", case=False, na=False)
    ].sort_values(["Accuracy", "F1 Score", "ROC AUC"], ascending=False)
)

In [ ]:
# Install Optuna
!pip -q install optuna

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


In [ ]:
X_train_t = preprocess.fit_transform(train_df[X_cols])
X_valid_t = preprocess.transform(valid_df[X_cols])

y_train = train_df[y_col].astype(int).values
y_valid = valid_df[y_col].astype(int).values


In [ ]:
def tune_logistic(trial, penalty):
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    model = LogisticRegression(
        penalty=penalty,
        solver="saga",
        C=C,
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    aucs = []

    for train_idx, val_idx in cv.split(X_train_t, y_train):
        X_tr, X_va = X_train_t[train_idx], X_train_t[val_idx]
        y_tr, y_va = y_train[train_idx], y_train[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, probs))

    return np.mean(aucs)


In [ ]:
study_l2 = optuna.create_study(direction="maximize")
study_l2.optimize(lambda trial: tune_logistic(trial, penalty="l2"), n_trials=100)

print("Best L2 AUROC:", study_l2.best_value)
print("Best L2 params:", study_l2.best_params)


In [ ]:
study_l1 = optuna.create_study(direction="maximize")
study_l1.optimize(lambda trial: tune_logistic(trial, penalty="l1"), n_trials=100)

print("Best L1 AUROC:", study_l1.best_value)
print("Best L1 params:", study_l1.best_params)


In [ ]:
def fit_and_eval(best_params, penalty, label):
    model = LogisticRegression(
        penalty=penalty,
        solver="saga",
        C=best_params["C"],
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    model.fit(X_train_t, y_train)
    probs = model.predict_proba(X_valid_t)[:, 1]
    auc = roc_auc_score(y_valid, probs)
    print(f"{label} Validation AUROC: {auc:.3f}")
    return model, probs


In [ ]:
# Check AUC after hyperparameter tuning
l2_model, l2_probs = fit_and_eval(study_l2.best_params, "l2", "L2 Logistic")
l1_model, l1_probs = fit_and_eval(study_l1.best_params, "l1", "LASSO Logistic")

In [ ]:
coef = l1_model.coef_.ravel()
nonzero = (coef != 0).sum()
total = coef.shape[0]

print(f"Nonzero predictors: {nonzero} / {total}")


In [ ]:
# Extract feaure names by importance
feature_names = preprocess.get_feature_names_out()
coef = l1_model.coef_.ravel()

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coef})
      .assign(abs_coef=lambda d: d.coef.abs())
      .query("coef != 0")
      .sort_values("abs_coef", ascending=False)
)

coef_df.head(50)


In [ ]:
coef_df_nz["cum_mass"] = coef_df_nz["abs_coef"].cumsum() / coef_df_nz["abs_coef"].sum()

k80 = int((coef_df_nz["cum_mass"] <= 0.80).sum() + 1)
k90 = int((coef_df_nz["cum_mass"] <= 0.90).sum() + 1)

print("Predictors to reach 80% of |coef| mass:", k80)
print("Predictors to reach 90% of |coef| mass:", k90)

display(
    coef_df_nz.loc[k90-1:k90+1, ["feature","coef","abs_coef","cum_mass"]]
)


In [ ]:
# Print ROC curve and Confusion Matrix for the final selected model
#The threshold used to calculate the classification metrics is 0.5
#(default threshold) by the predict method of LogisticRegression to convert
#predicted probabilities into binary class labels.

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, RocCurveDisplay, confusion_matrix, roc_auc_score, f1_score, balanced_accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay
import numpy as np

# --- ROC Curve for LASSO Logistic Regression ---
plt.figure(figsize=(8, 6))
fpr_l1, tpr_l1, _ = roc_curve(y_valid, l1_probs)
roc_auc_l1 = roc_auc_score(y_valid, l1_probs)

display_l1 = RocCurveDisplay(fpr=fpr_l1, tpr=tpr_l1, roc_auc=roc_auc_l1, estimator_name='LASSO Logistic')
display_l1.plot(ax=plt.gca())

plt.title('ROC Curve for LASSO Logistic Regression')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level (AUC = 0.5)')
plt.legend()
plt.grid(True)
plt.show()

# --- Confusion Matrix and Performance Stats for LASSO Logistic Regression ---

y_pred_l1 = l1_model.predict(X_valid_t)
cm_l1 = confusion_matrix(y_valid, y_pred_l1)

print("\n--- Confusion Matrix for LASSO Logistic Regression ---")
print(cm_l1)
ConfusionMatrixDisplay(cm_l1, display_labels=[0, 1]).plot()
plt.title('Confusion Matrix: LASSO Logistic')
plt.show()

# Extract values from confusion matrix
# cm_l1 is typically [[TN, FP], [FN, TP]]
TN, FP, FN, TP = cm_l1.ravel()

# Calculate Sensitivity (Recall)
recall = recall_score(y_valid, y_pred_l1)

# Calculate Specificity
specificity = TN / (TN + FP)

# Calculate F1 Score
f1 = f1_score(y_valid, y_pred_l1)

# Calculate Balanced Accuracy
balanced_accuracy = balanced_accuracy_score(y_valid, y_pred_l1)

# Calculate Precision
precision = precision_score(y_valid, y_pred_l1)

print("\n--- Performance Statistics for LASSO Logistic Regression ---")
print(f"ROC AUC: {roc_auc_l1:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")

In [ ]:
# Get top 50 feature names from the preprocessor
all_feature_names = preprocess.get_feature_names_out()

# Create lasso_freq DataFrame
lasso_freq = pd.DataFrame({
    "feature": all_feature_names,
    "selection_pct": 0.0  # Default to 0.0, will update for selected features
})

# Update selection_pct for features present in coef_df_nz (selected by LASSO)
selected_features = coef_df_nz['feature'].tolist()
lasso_freq.loc[lasso_freq['feature'].isin(selected_features), 'selection_pct'] = 1.0

# Now, the 'imp' DataFrame can be created
imp = (lasso_freq.merge(coef_df_nz, on="feature", how="inner")
       .assign(stability_abs=lambda d: d["selection_pct"] * d["abs_coef"])
       .sort_values("stability_abs", ascending=False)
       .reset_index(drop=True)
)

display(imp.head(50)[["feature","coef","abs_coef","selection_pct","stability_abs"]])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Use your tuned params
C_best = study_l1.best_params["C"]  # adjust if your object stores differently

lasso = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=C_best,
    class_weight="balanced",   # or None if you didn't use it
    max_iter=10000, # Increased max_iter to improve convergence
    random_state=1234
)

final_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", lasso)
])

final_pipe.fit(train_df[X_cols], train_df[y_col].astype(int))

In [ ]:
valid_df = valid_df.copy()

valid_df["p_utd"] = final_pipe.predict_proba(valid_df[X_cols])[:, 1]
valid_df["p_hat"] = 1 - valid_df["p_utd"]   # risk of NOT being UTD


In [ ]:
# Crea risk-tiers
valid_df["risk_tier"] = pd.cut(
    valid_df["p_hat"],
    bins=5,
    labels=["Tier 1 (Lowest)","Tier 2","Tier 3","Tier 4","Tier 5 (Highest)"]
)

In [ ]:
# Oberved/Actual UTD by risk-tiers
tier_table = (
    valid_df.groupby("risk_tier")
    .agg(
        n=("risk_tier","size"),
        mean_risk=("p_hat","mean"),
        min_risk=("p_hat","min"),
        max_risk=("p_hat","max"),
        observed_utd=(y_col,"mean"),
        std_observed_utc=(y_col,"std")
    )
    .reset_index()
)
tier_table["observed_not_utd"] = 1 - tier_table["observed_utd"]
tier_table["pct_pop"] = tier_table["n"] / tier_table["n"].sum()

fmt = {
    "mean_risk": "{:.4f}",
    "min_risk": "{:.4f}",
    "max_risk": "{:.4f}",
    "observed_utd": "{:.1%}",
    "observed_not_utd": "{:.1%}",
    "pct_pop": "{:.1%}",
    "std_observed_utc": "{:.4f}"
}
display(tier_table.style.format(fmt))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# REVERSED palette: green → yellow → red
palette = sns.color_palette("RdYlGn_r", n_colors=len(tier_table))

# Sort by observed UTD (highest first)
tier_table_sorted = tier_table.sort_values('observed_utd', ascending=False)

ax = sns.barplot(
    x='risk_tier',
    y='observed_utd',
    data=tier_table_sorted,
    palette=palette,
    order=tier_table_sorted['risk_tier']
)

# --- NEW: custom tier labels ---
tier_labels = ["Very Low", "Low", "Moderate", "High", "Very High"]
ax.set_xticklabels(tier_labels)

plt.title('Observed UTD Proportion by Risk Tier')
plt.xlabel('Predicted Risk Tiers')   # UPDATED
plt.ylabel('Observed UTD Proportion')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add text labels
for i, (_, row) in enumerate(tier_table_sorted.iterrows()):
    ax.text(
        i,
        row['observed_utd'],
        f'{row["observed_utd"] * 100:.1f}% UTD',
        ha='center',
        va='bottom'
    )

plt.show()


In [ ]:
# Align with SEM Domains (modify as needed further)

def sem_bucket(feature: str) -> str:
    f = feature.upper()

    # Organizational / healthcare delivery
    if any(k in f for k in ["WELLCHILD", "CKUP", "VISITS", "HPVI_RECOM", "FACILITY"]):
        return "Organizational (healthcare access/delivery)"

    # Community / geography / structural environment
    if any(k in f for k in ["STATE_", "CEN_REG", "MOBIL_"]):
        return "Community / Geography (structural context)"

    # SES resources
    if any(k in f for k in ["INCPOV", "INCQ"]):
        return "Socioeconomic resources"

    # Interpersonal / family
    if any(k in f for k in ["C5R_", "MARITAL", "LANGUAGE", "CHILDNM"]):
        return "Interpersonal / Family"

    # Individual
    if any(k in f for k in ["RACE", "RACEETH", "CPOX", "RISK_NOW"]):
        return "Individual"

    return "Other / check mapping"


top50 = coef_df_nz.head(50).copy()
top50["SEM_domain"] = top50["feature"].apply(sem_bucket)

display(
    top50[["SEM_domain", "feature", "coef", "abs_coef"]]
    .sort_values(["SEM_domain", "abs_coef"], ascending=[True, False])
    .style.format({"coef":"{:.2f}", "abs_coef":"{:.2f}"})
)


In [ ]:
# Importance of domains
sem_mass = (
    top50.groupby("SEM_domain")["abs_coef"].sum()
    .sort_values(ascending=False)
    .to_frame("abs_coef_mass")
)
sem_mass["pct_mass"] = sem_mass["abs_coef_mass"] / sem_mass["abs_coef_mass"].sum()

display(sem_mass.style.format({"abs_coef_mass":"{:.3f}", "pct_mass":"{:.1%}"}))


In [ ]:
rows = []

for tier in sorted(tier_features.keys()):
    for direction in ["positive", "negative"]:
        df = tier_features[tier][direction].copy()
        df["risk_tier"] = tier
        df["direction"] = "Higher UTD" if direction == "positive" else "Lower UTD"
        rows.append(df)

tier_feature_table = pd.concat(rows, ignore_index=True)

# Optional: clean feature names for readability
tier_feature_table["feature_clean"] = (
    tier_feature_table["feature"]
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ")
)

display(
    tier_feature_table
    .sort_values(["risk_tier", "direction", "mean_contribution"], ascending=[True, True, False])
    .style.format({"mean_contribution": "{:.1%}"})
)
